# Getting Started: Concepts & Abstraction Layers
*Fabian Nütz*

This notebook explains the conceptual model behind pandaict. By the end you will understand:

- The five core element types and which network layer each belongs to
- How pandaict's abstraction maps to the ISO/OSI model
- What a `pandaICTNet` object is and how it stores data
- How elements are identified (index tuples and UUIDs)
- The role of **standard types** and **enums**

## Setup

In [1]:
import pandaict as pi
from pandaict.utils.enums import NodeType, DeviceType, AppType, InfoItem

## The pandaICT network object

Every ICT network modeling session starts with `create_empty_network()`:

In [2]:
net = pi.create_empty_network(name='My First Network')
print(type(net).__name__)   # pandaICTNet
print(net.name)             # 'My First Network'

pandaICTNet
My First Network


A `pandaICTNet` behaves like a Python `dict` with attribute access. It contains **five pandas DataFrames**: one per element type plus a `std_types` dictionary and a few metadata entries:

In [3]:
print('DataFrame slots:', [k for k in net.keys() if k in ('node','link','device','app','conn')])
print('Other keys:    ', [k for k in net.keys() if k not in ('node','link','device','app','conn')])

DataFrame slots: ['node', 'link', 'device', 'app', 'conn']
Other keys:     ['name', 'connectors', 'iptables', 'std_types']


## Five element types mapped to ISO/OSI layers

| Element | pandaict Layer | ISO/OSI layer | Description |
|---------|-------|-------|-------------|
| **node**   | Physical | 1–3 | Network interconnection point (switch, router). Supports one or more link types. |
| **device** | Physical | 1–3 | Communication endpoint (server, RTU, converter). Hosts apps. |
| **link**   | Physical | 1–3 | Point-to-point physical connection between nodes or device–node. |
| **app**    | Application | 4–7  | Application-level service running on a device. Communication peer. |
| **conn**   | Application & Information | 4–7  | End-to-end logical connection between two apps, using a specific protocol to transmit information items. |

### Physical layer (ISO/OSI 1–3): node, device, link

The physical layer models **hardware topology**: which devices and nodes exist, how they are wired together, and what physical technologies (link types) the connections use.

In [4]:
# Create a minimal physical layer
nA = pi.create_ict_node(net, name='NodeA', node_type=NodeType.SWITCH, link_types=['LAN1000'])
nB = pi.create_ict_node(net, name='NodeB', node_type=NodeType.SWITCH, link_types=['LAN1000'])
d1 = pi.create_ict_device(net, name='Alice', device_type=DeviceType.SERVER)
d2 = pi.create_ict_device(net, name='Bob',   device_type=DeviceType.SERVER)

pi.create_ict_link(net, name='L-AB', from_elt=nA, to_elt=[nB], link_type='LAN1000')
pi.create_ict_link(net, name='L-A1', from_elt=d1, to_elt=[nA], link_type='LAN1000')
pi.create_ict_link(net, name='L-B2', from_elt=d2, to_elt=[nB], link_type='LAN1000')

print('nodes:', len(net.node), '| devices:', len(net.device), '| links:', len(net.link))

nodes: 2 | devices: 2 | links: 3


**Key rules for links**:
- A link has exactly one `from_elt` and a **list** of `to_elt`s (multicast support).
- The `from -> to` direction is always called **downstream**, the reverse direction is called **upstream**.
- Direct device-to-device links are generally **not allowed** — place a dummy node between them if needed.

### Application layer (ISO/OSI 4–7): app, conn

The application layer models **end-to-end communication**: which apps talk to each other, using which protocol, and what information items they exchange.

In [5]:
a1 = pi.create_ict_app(net, name='App-Alice', app_type=AppType.CLIENT,   device=d1)
a2 = pi.create_ict_app(net, name='App-Bob',   app_type=AppType.WEBSERVER, device=d2)

pi.create_ict_connection(net, name='Conn-AB', from_app=a1, to_app=a2,
                         protocol='TCP')

print('apps:', len(net.app), '| connections:', len(net.conn))

apps: 2 | connections: 1


**Key rules for connections**:
- One app is associated with exactly one device.
- One device may host multiple apps.
- Every connection uses a single protocol.

### Information layer: info_items

This pandaict layer does not add new element types, but adds definition about the information being transmitted through connections, using info items. 

In [6]:
# Delete connection and re-create with info item definition
net.conn.drop(0, inplace=True) 
pi.create_ict_connection(net, name='Conn-AB', from_app=a1, to_app=a2,
                         protocol='TCP',
                         info_items=[InfoItem.REQUEST, InfoItem.RESPONSE],
                         peers=(a1, a2),
                         producer=[0, 1])   # a1 produces REQUEST, a2 produces RESPONSE
print('apps:', len(net.app), '| connections:', len(net.conn))

apps: 2 | connections: 1


**Key rules for information items**
- Each connection can transmit an arbitraty number of info items 
- The protocol needs to support transmission of all info items defined, as specified by the protocol type
- The `peers` / `producer` mechanism tracks which peers exchange the info items, and which peer produces which info item — critical for routing and data-flow modeling.

## Element identification

Every element is identified in one of two ways:

| Style | Example | When to use |
|-------|---------|-------------|
| **Index tuple** | `('node', 0)` | Default — compact, human-readable |
| **UUID** | `uuid.UUID('...')` | Globally unique, serialization-safe |

Index tuples are `(element_type_string, integer_index)` where the index is the row number in that element's DataFrame.

In [7]:
print('nA =', nA)                    # ('node', 0)
print('nB =', nB)                    # ('node', 1)
print()
print('node DataFrame index:')
print(net.node.index.to_list())     # [0, 1]

nA = ('node', 0)
nB = ('node', 1)

node DataFrame index:
[0, 1]


In [8]:
# Create a node using UUID mode
uid_node = pi.create_ict_node(net, name='Node-UUID', node_type=NodeType.ROUTER,
                              link_types=['LAN1000'], use_uuid=True)
print('UUID return:', uid_node)      # e.g. UUID('0a004f60-...')
print(type(uid_node))               # <class 'uuid.UUID'>

UUID return: f86ec32c-7c35-43c0-afae-31103cfbfa65
<class 'uuid.UUID'>


## Standard types and enums

Two mechanisms define the **vocabulary** of your network model:

| Mechanism | Storage | Examples |
|-----------|---------|----------|
| **std_types** | `net.std_types['link']`, `net.std_types['protocol']` | link types (e.g. LAN1000, LTE50), protocol types (e.g. TCP, MODBUS) |
| **Enums** | `NodeType`, `DeviceType`, `AppType`, `InfoItem` | SWITCH, SERVER, CLIENT, REQUEST |

Enums are implemented with `aenum`. Standard types are plain dictionaries with required fields like `bandwidth_Mbps`, `latency_ms`, `mtbf_h`, `mttr_h` for links. The pre-defined set is very limited, but all link types, protocol types and enums are extensible at runtime using create_std_type(). 

In [10]:
print('Built-in link types:', list(net.std_types['link'].keys()))
print()
print('LAN1000 spec:', net.std_types['link']['LAN1000'])

Built-in link types: ['Binary', 'RS485', 'LAN1000', 'LTE50', 'UMTS', 'ADSL8', 'Fiber']

LAN1000 spec: {'bandwidth_Mbps': (1000, 1000), 'latency_ms': 100, 'wired': True, 'protocols': ['TCP', 'UDP', 'FTP', 'HTTP', 'HTTPS', 'MODBUS-TCP'], 'ber_per_km': 2e-08, 'mtbf_h': 8000, 'mttr_h': 2, 'uuid': UUID('4d5b388f-ce1c-407e-81c6-5caa23f1832b')}


## Detailed descriptions

| Concept | Notebook |
|---------|----------|
| Physical layer details (nodes, devices, links) | **02** |
| Logical layer details (apps, connections, converters) | **03** |
| Calculations and workflows | **04** |
| Inspection and debugging | **05** |
| Customizing standard types and enums | **06** |
| Advanced topics | **07** |

Next: [02 – Physical Layer](02_physical_layer.ipynb)